# Phase 7 Analysis: Exchange Registration (COMEX)

## Objective

Analyze gold inventory at the exchange level (Phase 7), our highest-transparency anchor point.

This notebook examines:
- Daily inventory levels (registered vs eligible)
- Flow patterns and volatility
- Warehouse-level distribution
- Long-term trends

## Phase 7 Context

**Physical State**: Deliverable bullion (exchange-registered bars)

**Transformation**: Legal/registry status change

**Transparency**: HIGH - Daily public reporting

**Why This Matters**: COMEX inventory is the terminus of the physical supply chain before paper claims dominate. It's our reconciliation target for upstream flow analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

# Notebook configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

%matplotlib inline


## 1. Data Loading


In [ ]:
# Load schema definitions
schema_dir = Path("../../schema")
data_dir = Path("../../data")

phases = pd.read_csv(schema_dir / "supply_chain_phases.csv")
metrics = pd.read_csv(schema_dir / "gold_supply_chain_metrics.csv")

# Filter for Phase 7 data
phase7_data = metrics[metrics['phase_id'] == 7].copy()

print(f"Phase 7 records found: {len(phase7_data)}")
if len(phase7_data) > 0:
    print(f"Date range: {phase7_data['date'].min()} to {phase7_data['date'].max()}")


In [ ]:
# Examine data structure
if len(phase7_data) > 0:
    display(phase7_data.head())
else:
    print("⚠️ No Phase 7 data loaded yet. Run COMEX scraper first.")


## 2. Inventory Overview

### Key Metrics:
- **Registered**: Available for immediate delivery against futures contracts
- **Eligible**: Meets exchange standards but not committed for delivery
- **Total**: Sum of registered + eligible


In [ ]:
# Summary statistics
if len(phase7_data) > 0:
    summary = phase7_data.groupby('metric_name')['metric_value'].describe()
    print("Inventory Statistics (troy ounces):")
    display(summary)


## 3. Export for Frontend

Phase 7 summary exported as JSON for the interactive simulator.


In [ ]:
# Export summary for frontend consumption
import json

if len(phase7_data) > 0:
    output = {
        'phase_id': 7,
        'phase_name': 'Exchange Registration (COMEX)',
        'transparency': 'High',
        'last_update': str(phase7_data['date'].max()),
        'total_records': len(phase7_data),
        'data_quality': 'High',
        'source': 'CME Group'
    }
    
    output_file = data_dir / 'processed' / 'phase7_summary.json'
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w') as f:
        json.dump(output, f, indent=2)
    
    print(f"✓ Exported summary to {output_file}")
    print(json.dumps(output, indent=2))
else:
    print("⚠️ No data to export. Run data collection first.")
